# PATSTAT Tutorial: Find Your University's Patents

**Target audience:** Patent information professionals with no SQL experience

**Example:** TU Berlin (easily replaceable)

**Platform:** EPO Technology Intelligence Platform (TIP)

**Edition:** PATSTAT Global, Autumn 2025

**Created:** 2026-03-02 | mtc.berlin for EPO Academy

---

### How This Tutorial Works

Each query builds on the previous one. We start with the simplest question ("What is my university called in PATSTAT?") and work our way up to family analysis.

### To Customize

1. Replace `'TECHNISCHE UNIVERSITAET BERLIN'` with the PSN name of your university (you'll find it with Query 1)
2. Adjust the year range (`BETWEEN 2000 AND 2024`)

### Key PATSTAT Concepts

<div align="left">

| Term | Meaning |
|:-----|:--------|
| `person_name` | Name as delivered by the patent office (many variants!) |
| `psn_name` | PATSTAT-standardized name (best choice for universities) |
| `han_name` | OECD-harmonized name (has known errors!) |
| `docdb_family_id` | One invention, regardless of how many countries it was filed in |
| `applt_seq_nr > 0` | Filter for applicants only (not inventors) |

</div>

## Setup: Connect to PATSTAT

In [1]:
from epo.tipdata.patstat import PatstatClient
import pandas as pd
import time

# Connect to PATSTAT
patstat = PatstatClient(env='PROD')

def run_query(query):
    """Execute a PATSTAT SQL query and return a pandas DataFrame."""
    start = time.time()
    res = patstat.sql_query(query, use_legacy_sql=False)
    df = pd.DataFrame(res)
    print(f"Query took {time.time() - start:.1f}s - {len(df)} rows returned")
    return df

---

## Query 1: Find Name Variants

**Why:** In PATSTAT, every organization exists under different spellings. Before you can analyze anything, you need to know how your university appears in the data.

**Tip:** Start with a broad search term (e.g. `'%berlin%'` and `'%technische%'`) and see what comes back.

**Try it:** Change `SEARCH_TERM` to match your university.

In [2]:
# --- CHANGE THIS ---
SEARCH_TERM = '%technische%berlin%'
# -------------------

df_q1 = run_query(f"""
SELECT 
    p.person_name,
    p.psn_name,
    p.han_name,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND (   LOWER(p.person_name) LIKE '{SEARCH_TERM}'
       OR LOWER(p.han_name)    LIKE '{SEARCH_TERM}')
  AND a.docdb_family_id > 0
GROUP BY p.person_name, p.psn_name, p.han_name
HAVING COUNT(DISTINCT a.docdb_family_id) >= 3
ORDER BY families DESC
LIMIT 20
""")

df_q1

Query took 2.9s - 20 rows returned


,person_name,psn_name,han_name,families
0,Technische Universität Berlin,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAT MUNCHEN,340
1,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAT MUNCHEN,218
2,TECHNISCHE UNIVERSITÄT BERLIN,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAT MUNCHEN,113
3,Technische Universitat Berlin,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAT MUNCHEN,70
4,Technische Universitaet Berlin,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAT MUNCHEN,50
5,TECHNISCHE UNIVERSITAT BERLIN,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAT BERLIN,42
6,"VEB LUFTTECHNISCHE ANLAGEN BERLIN,DD",VEB LUFTTECHNISCHE ANLAGEN BERLIN,VEB LUFTTECHNISCHE ANLAGEN BERLIN DD,21
7,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAET BERLIN,TECHNISCHE UNIVERSITAET BERLIN,20
8,PHYSIKALISCH-TECHNISCHE BUNDESANSTALT BRAUNSCH...,PHYSIKALISCH-TECHNISCHE BUNDESANSTALT BRAUNSCH...,PHYSIKALISCH TECHNISCHE BUNDESANSTALT BRAUNSCH...,14
9,PAUL JORDAN ELEKTROTECHNISCHE FABRIK GMBH & CO...,PAUL JORDAN ELEKTROTECHNISCHE FABRIK & COMPANY...,PAUL JORDAN ELEKTROTECHNISCHE FABRIK GMBH & CO,14


### What to Look For

You will likely see multiple rows for the same university with different spellings in `person_name`, but the same `psn_name`. For example:

- `Technische Universität Berlin` -> `TECHNISCHE UNIVERSITAET BERLIN`
- `TECHNISCHE UNIVERSITAET BERLIN` -> `TECHNISCHE UNIVERSITAET BERLIN`
- `TECHNISCHE UNIVERSITÄT BERLIN` -> `TECHNISCHE UNIVERSITAET BERLIN`

**Warning:** `han_name` may show incorrect values! For TU Berlin, it shows "TECHNISCHE UNIVERSITAT MUNCHEN" — this is a known PATSTAT error where the OECD harmonization merged TU Berlin and TU Munich. **Always use `psn_name` for universities.**

**Remember:** `psn_name = 'TECHNISCHE UNIVERSITAET BERLIN'` is the filter we'll use from now on.

---

## Query 2: Portfolio Overview – How Big Is It?

**Why:** Before diving into details, you want to know the basic numbers. How many inventions (families)? How many individual filings? Since when?

In [3]:
# --- CHANGE THIS ---
PSN_NAME = 'TECHNISCHE UNIVERSITAET BERLIN'
# -------------------

df_q2 = run_query(f"""
SELECT 
    COUNT(DISTINCT a.docdb_family_id) AS patent_families,
    COUNT(DISTINCT a.appln_id)        AS individual_filings,
    MIN(a.appln_filing_year)          AS first_filing,
    MAX(a.appln_filing_year)          AS last_filing
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 1990 AND 2024
""")

df_q2

Query took 2.6s - 1 rows returned


,patent_families,individual_filings,first_filing,last_filing
0,507,1268,1995,2024


### How to Read This

For TU Berlin you would expect something like: ~507 inventions turned into ~1,268 filings worldwide. That means each invention was filed in ~2.5 countries on average.

---

## Query 3: Timeline – How Is Activity Developing?

**Why:** Spot trends. Is patent activity rising or declining? Are there notable years?

**Note:** The most recent 2 years (2023–2024) are likely incomplete due to publication delays.

In [4]:
df_q3 = run_query(f"""
SELECT 
    a.appln_filing_year               AS year,
    COUNT(DISTINCT a.docdb_family_id) AS families,
    COUNT(DISTINCT a.appln_id)        AS filings
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 2000 AND 2024
GROUP BY a.appln_filing_year
ORDER BY a.appln_filing_year
""")

df_q3

Query took 2.3s - 25 rows returned


,year,families,filings
0,2000,6,7
1,2001,8,18
2,2002,8,14
3,2003,10,12
4,2004,15,19
5,2005,25,31
6,2006,23,32
7,2007,40,53
8,2008,33,45
9,2009,38,73


---

## Query 4: A Patent Family Under the Microscope

**Why:** Now it gets concrete! We take the largest family and look at all its members.

### What Is a Patent Family?

A DOCDB family groups all filings that protect the **same invention** — regardless of which country they were filed in.

- **EP** = European Patent
- **WO** = PCT (international) application
- **DE** = Germany, **US** = USA, **CN** = China, etc.

The query first finds the largest family, then displays all its members.

In [5]:
df_q4 = run_query(f"""
WITH uni_apps AS (
    SELECT DISTINCT
        a.appln_id,
        a.docdb_family_id
    FROM tls201_appln a
    JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
    JOIN tls206_person p ON pa.person_id = p.person_id
    WHERE pa.applt_seq_nr > 0
      AND p.psn_name = '{PSN_NAME}'
      AND a.docdb_family_id > 0
),
largest_family AS (
    SELECT docdb_family_id, COUNT(*) AS family_size
    FROM uni_apps
    GROUP BY docdb_family_id
    ORDER BY family_size DESC
    LIMIT 1
)
SELECT 
    lf.docdb_family_id,
    lf.family_size       AS total_members,
    a.appln_auth          AS authority,
    a.appln_nr            AS application_number,
    a.appln_filing_date   AS filing_date,
    a.granted,
    t.appln_title         AS title
FROM tls201_appln a
JOIN largest_family lf ON a.docdb_family_id = lf.docdb_family_id
LEFT JOIN tls202_appln_title t 
    ON a.appln_id = t.appln_id AND t.appln_title_lg = 'en'
WHERE a.docdb_family_id > 0
ORDER BY a.appln_filing_date, a.appln_auth
""")

df_q4

Query took 4.1s - 24 rows returned


,docdb_family_id,total_members,authority,application_number,filing_date,granted,title
0,39734907,20,EP,08153596,2008-03-28,N,Methods for producing de novo papillae and hai...
1,39734907,20,AT,09724502,2009-03-23,Y,None
2,39734907,20,AU,2009228756,2009-03-23,Y,Methods for producing hair microfollicles and ...
3,39734907,20,CA,2719769,2009-03-23,Y,METHODS FOR PRODUCING HAIR MICROFOLLICLES AND ...
4,39734907,20,CN,200980116851,2009-03-23,Y,Methods for producing hair microfollicles and ...
5,39734907,20,DK,09724502,2009-03-23,Y,None
6,39734907,20,EP,09724502,2009-03-23,Y,METHODS FOR PRODUCING HAIR MICROFOLLICLES AND ...
7,39734907,20,EP,12150985,2009-03-23,Y,Methods for producing hair microfollicles and ...
8,39734907,20,ES,09724502,2009-03-23,Y,None
9,39734907,20,ES,12150985,2009-03-23,Y,None


### How to Read This

An invention filed in 15+ countries signals high commercial significance. Look at:
- Which offices received filings (EP, WO, US, CN, JP...)
- Whether it was granted (Y/N) at each office
- The English title to understand what the invention is about

---

## Query 5: Filing Strategy – Which Countries?

**Why:** The filing strategy shows where the university wants to commercially exploit its inventions.

In [6]:
df_q5 = run_query(f"""
SELECT 
    a.appln_auth AS authority,
    COUNT(DISTINCT a.docdb_family_id) AS families,
    COUNT(DISTINCT a.appln_id)        AS filings,
    SUM(CASE WHEN a.granted = 'Y' THEN 1 ELSE 0 END) AS granted
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 2000 AND 2024
GROUP BY a.appln_auth
ORDER BY families DESC
LIMIT 15
""")

df_q5

Query took 2.6s - 15 rows returned


,authority,families,filings,granted
0,EP,277,317,126
1,WO,263,263,10
2,DE,241,252,81
3,US,198,212,161
4,CN,49,49,33
5,ES,34,35,35
6,CA,33,33,17
7,AU,18,18,8
8,KR,12,13,9
9,DK,11,11,11


### How to Interpret

For a typical German university:
- **EP + WO** dominate → The European and international (PCT) systems are the standard filing route
- **US** with high grant rate → Quality portfolio aimed at the US market
- **CN** selective → Only the most important inventions go to China
- **DE** (national) → Domestic filings, often the initial priority filing

---

## Query 6: Technology Profile – Which Fields?

**Why:** The technology profile reveals research strengths. We use the WIPO 35 technology fields for a clear overview.

In [7]:
df_q6 = run_query(f"""
SELECT 
    tf.techn_sector    AS sector,
    tf.techn_field     AS technology_field,
    COUNT(DISTINCT a.docdb_family_id) AS families
FROM tls201_appln a
JOIN tls207_pers_appln pa ON a.appln_id = pa.appln_id
JOIN tls206_person p ON pa.person_id = p.person_id
JOIN tls230_appln_techn_field atf ON a.appln_id = atf.appln_id
JOIN tls901_techn_field_ipc tf ON atf.techn_field_nr = tf.techn_field_nr
WHERE pa.applt_seq_nr > 0
  AND p.psn_name = '{PSN_NAME}'
  AND a.docdb_family_id > 0
  AND a.appln_filing_year BETWEEN 2000 AND 2024
GROUP BY tf.techn_sector, tf.techn_field
ORDER BY families DESC
LIMIT 10
""")

df_q6

Query took 2.6s - 10 rows returned


,sector,technology_field,families
0,Electrical engineering,Computer technology,72
1,Electrical engineering,Digital communication,59
2,Instruments,Optics,56
3,Instruments,Measurement,51
4,Instruments,Medical technology,48
5,Chemistry,Chemical engineering,46
6,Electrical engineering,"Electrical machinery, apparatus, energy",45
7,Electrical engineering,Semiconductors,44
8,Electrical engineering,Audio-visual technology,40
9,Chemistry,Biotechnology,33


### How to Read This

TU Berlin is a broadly positioned technical university. You would expect:
- Strong in electrical engineering/IT (computer technology, digital communication, semiconductors)
- Significant in measurement and optics
- Surprisingly strong in medical technology and biotechnology

---

## Try It Yourself!

Replace `PSN_NAME` in the second query above. Here are some examples:

<div align="left">

| University | `psn_name` |
|:-----------|:-----------|
| TU Munich | `TECHNISCHE UNIVERSITAET MUENCHEN` |
| KIT Karlsruhe | `KARLSRUHER INSTITUT FUER TECHNOLOGIE` |
| RWTH Aachen | `RHEINISCH WESTFAELISCHE TECHNISCHE HOCHSCHULE AACHEN` |
| TU Dresden | `TECHNISCHE UNIVERSITAET DRESDEN` |
| FU Berlin | `FREIE UNIVERSITAET BERLIN` |
| HU Berlin | `HUMBOLDT-UNIVERSITAET ZU BERLIN` |

</div>

**Not sure about the name?** Always start with Query 1 and search broadly (e.g. `'%karlsruhe%'`).

---

### Further Topics (next tutorials)

- **Inventor analysis:** Who are the top inventors?
- **Co-applicants:** Who does the university collaborate with?
- **Citation analysis:** Which inventions are the most influential?
- **IPC/CPC deep dive:** Detailed technology profiles

---

*Created with PATSTAT BigQuery + patstat-mcp + Claude AI | mtc.berlin | depa.tech*